In [1]:
import pandas as pd
import numpy as np
from dep2pyodbc import dep2connection

pd.set_option("display.max_columns", None)

channel_crh = dep2connection("CRH")
channel_dwh_lisa = dep2connection("CRH_DWH")
cursor = channel_dwh_lisa.cursor()

pyodbc using windows
pyodbc using windows


In [2]:
paq = pd.read_csv('./csv/raw_data/decoded_paq.csv')
paq.head()

,Unnamed: 0,CandidateID,InstanceID,left_statement,right_statement,anwser_val
0,0,3811418,1,BEW_1,BEH_1,1.0
1,1,3811418,1,REA_1,OPT_1,5.0
2,2,3811418,1,REG_1,AFW_1,5.0
3,3,3811418,1,RUS_1,ENE_1,5.0
4,4,3811418,1,OVE_1,GED_1,5.0


In [3]:
df_paq = paq[['CandidateID', 'InstanceID', 'left_statement', 'right_statement', 'anwser_val']]
df_paq.rename(columns={
    'left_statement': 'LeftStatement',
    'right_statement': 'RightStatement',
    'anwser_val': 'AnswerVal'
    }, inplace=True)
df_paq['Test'] = 'PAQ'
df_paq.head()

C:\Users\monad\AppData\Local\Temp\ipykernel_26600\1541107905.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_paq.rename(columns={


,CandidateID,InstanceID,LeftStatement,RightStatement,AnswerVal,Test
0,3811418,1,BEW_1,BEH_1,1.0,PAQ
1,3811418,1,REA_1,OPT_1,5.0,PAQ
2,3811418,1,REG_1,AFW_1,5.0,PAQ
3,3811418,1,RUS_1,ENE_1,5.0,PAQ
4,3811418,1,OVE_1,GED_1,5.0,PAQ


In [4]:
df_paq.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 955148 entries, 0 to 955147
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   CandidateID     955148 non-null  int64  
 1   InstanceID      955148 non-null  int64  
 2   LeftStatement   955148 non-null  object 
 3   RightStatement  955148 non-null  object 
 4   AnswerVal       955147 non-null  float64
 5   Test            955148 non-null  object 
dtypes: float64(1), int64(2), object(3)
memory usage: 43.7+ MB


### Candidates 

In [5]:
df_candidates_before_key = pd.read_sql("SELECT ID, CandidateID, InstanceID, CreatedDate, ModifiedDate, VersionNumber FROM CandidateResultPAQ", channel_crh)
df_candidates_before_key.head()

C:\Users\monad\AppData\Local\Temp\ipykernel_26600\2065093392.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_candidates_before_key = pd.read_sql("SELECT ID, CandidateID, InstanceID, CreatedDate, ModifiedDate, VersionNumber FROM CandidateResultPAQ", channel_crh)


,ID,CandidateID,InstanceID,CreatedDate,ModifiedDate,VersionNumber
0,1,3811418,1,2019-01-23 20:26:36.960,2019-01-23 20:38:11.807,v1.0
1,2,3804633,1,2019-01-10 07:25:31.700,2019-01-10 07:35:43.023,v1.0
2,3,3811419,1,2019-01-28 14:15:59.637,2019-01-28 14:46:29.963,v1.0
3,4,3820114,1,2019-02-06 13:02:37.200,2019-02-06 13:03:58.663,v1.0
4,5,3833734,1,2019-02-27 09:37:49.690,2019-02-27 09:47:16.393,v1.0


In [6]:
df_paq = pd.merge(df_paq, df_candidates_before_key, on=["CandidateID", "InstanceID"], how="left")
df_paq.head()

,CandidateID,InstanceID,LeftStatement,RightStatement,AnswerVal,Test,ID,CreatedDate,ModifiedDate,VersionNumber
0,3811418,1,BEW_1,BEH_1,1.0,PAQ,1,2019-01-23 20:26:36.960,2019-01-23 20:38:11.807,v1.0
1,3811418,1,REA_1,OPT_1,5.0,PAQ,1,2019-01-23 20:26:36.960,2019-01-23 20:38:11.807,v1.0
2,3811418,1,REG_1,AFW_1,5.0,PAQ,1,2019-01-23 20:26:36.960,2019-01-23 20:38:11.807,v1.0
3,3811418,1,RUS_1,ENE_1,5.0,PAQ,1,2019-01-23 20:26:36.960,2019-01-23 20:38:11.807,v1.0
4,3811418,1,OVE_1,GED_1,5.0,PAQ,1,2019-01-23 20:26:36.960,2019-01-23 20:38:11.807,v1.0


In [7]:
dim_candidate = pd.read_sql("SELECT CandidateKey, ID, InstanceID FROM DimCandidate", channel_dwh_lisa)
dim_candidate.head()

C:\Users\monad\AppData\Local\Temp\ipykernel_26600\977585254.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_candidate = pd.read_sql("SELECT CandidateKey, ID, InstanceID FROM DimCandidate", channel_dwh_lisa)


,CandidateKey,ID,InstanceID
0,14,1,4
1,22,2,2
2,24,2,4
3,32,3,2
4,34,3,4


In [8]:
df_paq = pd.merge(df_paq, dim_candidate, left_on=["CandidateID", "InstanceID"], right_on=["ID", "InstanceID"], how="left")
df_paq.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 955148 entries, 0 to 955147
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   CandidateID     955148 non-null  int64         
 1   InstanceID      955148 non-null  int64         
 2   LeftStatement   955148 non-null  object        
 3   RightStatement  955148 non-null  object        
 4   AnswerVal       955147 non-null  float64       
 5   Test            955148 non-null  object        
 6   ID_x            955148 non-null  int64         
 7   CreatedDate     955148 non-null  datetime64[ns]
 8   ModifiedDate    955148 non-null  datetime64[ns]
 9   VersionNumber   955148 non-null  object        
 10  CandidateKey    857336 non-null  float64       
 11  ID_y            857336 non-null  float64       
dtypes: datetime64[ns](2), float64(3), int64(3), object(4)
memory usage: 87.4+ MB


In [9]:
df_paq.drop(columns=["ID_y", "CandidateID"], inplace=True)
df_paq.rename(columns={"ID_x": "TestID"}, inplace=True)
df_paq.head()

,InstanceID,LeftStatement,RightStatement,AnswerVal,Test,TestID,CreatedDate,ModifiedDate,VersionNumber,CandidateKey
0,1,BEW_1,BEH_1,1.0,PAQ,1,2019-01-23 20:26:36.960,2019-01-23 20:38:11.807,v1.0,38114181.0
1,1,REA_1,OPT_1,5.0,PAQ,1,2019-01-23 20:26:36.960,2019-01-23 20:38:11.807,v1.0,38114181.0
2,1,REG_1,AFW_1,5.0,PAQ,1,2019-01-23 20:26:36.960,2019-01-23 20:38:11.807,v1.0,38114181.0
3,1,RUS_1,ENE_1,5.0,PAQ,1,2019-01-23 20:26:36.960,2019-01-23 20:38:11.807,v1.0,38114181.0
4,1,OVE_1,GED_1,5.0,PAQ,1,2019-01-23 20:26:36.960,2019-01-23 20:38:11.807,v1.0,38114181.0


### Dates

In [10]:
df_dates = pd.read_sql("SELECT DateKey, Date FROM DimDate", channel_dwh_lisa)
df_dates.head()

C:\Users\monad\AppData\Local\Temp\ipykernel_26600\594440161.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dates = pd.read_sql("SELECT DateKey, Date FROM DimDate", channel_dwh_lisa)


,DateKey,Date
0,20070101,2007-01-01
1,20070102,2007-01-02
2,20070103,2007-01-03
3,20070104,2007-01-04
4,20070105,2007-01-05


In [11]:
df_paq["CreatedDate"] = pd.to_datetime(df_paq["CreatedDate"]).dt.date
df_paq = pd.merge(df_paq, df_dates, left_on="CreatedDate", right_on="Date", how="left")
df_paq.drop(columns=["Date", "CreatedDate"], inplace=True)
df_paq.rename(columns={"DateKey": "CreatedDateKey"}, inplace=True)

df_paq["ModifiedDate"] = pd.to_datetime(df_paq["ModifiedDate"]).dt.date
df_paq = pd.merge(df_paq, df_dates, left_on="ModifiedDate", right_on="Date", how="left")
df_paq.drop(columns=["Date", "ModifiedDate"], inplace=True)
df_paq.rename(columns={"DateKey": "ModifiedDateKey"}, inplace=True)

df_paq.head()

,InstanceID,LeftStatement,RightStatement,AnswerVal,Test,TestID,VersionNumber,CandidateKey,CreatedDateKey,ModifiedDateKey
0,1,BEW_1,BEH_1,1.0,PAQ,1,v1.0,38114181.0,20190123,20190123
1,1,REA_1,OPT_1,5.0,PAQ,1,v1.0,38114181.0,20190123,20190123
2,1,REG_1,AFW_1,5.0,PAQ,1,v1.0,38114181.0,20190123,20190123
3,1,RUS_1,ENE_1,5.0,PAQ,1,v1.0,38114181.0,20190123,20190123
4,1,OVE_1,GED_1,5.0,PAQ,1,v1.0,38114181.0,20190123,20190123


### Next TestKey available

In [12]:
df_test_sjt = pd.read_csv('../decoded_data/SJT/FactTest.csv')
df_test_sjt.head()

,Test,CandidateKey,CreatedDateKey,VersionNumber,TestKey,TimeSpent
0,SJT,NaN,NaN,v1.0,27,2915
1,SJT,NaN,NaN,v1.0,28,2048
2,SJT,NaN,NaN,v1.0,29,0
3,SJT,NaN,NaN,v1.0,30,0
4,SJT,NaN,NaN,V0.1,31,3962


In [13]:
max_key = df_test_sjt.TestKey.max()
max_key

6920

## Basic Cleaning

In [14]:
df_paq.info()
df_paq.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 955148 entries, 0 to 955147
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   InstanceID       955148 non-null  int64  
 1   LeftStatement    955148 non-null  object 
 2   RightStatement   955148 non-null  object 
 3   AnswerVal        955147 non-null  float64
 4   Test             955148 non-null  object 
 5   TestID           955148 non-null  int64  
 6   VersionNumber    955148 non-null  object 
 7   CandidateKey     857336 non-null  float64
 8   CreatedDateKey   955148 non-null  int64  
 9   ModifiedDateKey  955148 non-null  int64  
dtypes: float64(2), int64(4), object(4)
memory usage: 72.9+ MB


,InstanceID,LeftStatement,RightStatement,AnswerVal,Test,TestID,VersionNumber,CandidateKey,CreatedDateKey,ModifiedDateKey
0,1,BEW_1,BEH_1,1.0,PAQ,1,v1.0,38114181.0,20190123,20190123
1,1,REA_1,OPT_1,5.0,PAQ,1,v1.0,38114181.0,20190123,20190123
2,1,REG_1,AFW_1,5.0,PAQ,1,v1.0,38114181.0,20190123,20190123
3,1,RUS_1,ENE_1,5.0,PAQ,1,v1.0,38114181.0,20190123,20190123
4,1,OVE_1,GED_1,5.0,PAQ,1,v1.0,38114181.0,20190123,20190123


In [15]:
df_paq.dropna(inplace=True)

In [16]:
only_int = df_paq.AnswerVal.apply(lambda x: x % 1 == 0).all()
only_int

True

In [17]:
if only_int:
    df_paq['AnswerVal'] = df_paq['AnswerVal'].astype(int)
df_paq.info()

<class 'pandas.core.frame.DataFrame'>
Index: 857335 entries, 0 to 955146
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   InstanceID       857335 non-null  int64  
 1   LeftStatement    857335 non-null  object 
 2   RightStatement   857335 non-null  object 
 3   AnswerVal        857335 non-null  int32  
 4   Test             857335 non-null  object 
 5   TestID           857335 non-null  int64  
 6   VersionNumber    857335 non-null  object 
 7   CandidateKey     857335 non-null  float64
 8   CreatedDateKey   857335 non-null  int64  
 9   ModifiedDateKey  857335 non-null  int64  
dtypes: float64(1), int32(1), int64(4), object(4)
memory usage: 68.7+ MB


### Foutieve waarden

In [18]:
is_valid = df_paq['AnswerVal'].isin([1, 2, 3, 4, 5]).all()

In [19]:
if not is_valid: 
    print("Some values in the AnswerVal column do not fall in the permitted range.")
    df_paq = df_paq[df_paq['AnswerVal'].isin([1, 2, 3, 4, 5])]
else: 
    print("All values in the AnswerVal column fall in the permitted range.")

Some values in the AnswerVal column do not fall in the permitted range.


## FactTest

In [20]:
df_test = df_paq[["TestID", "Test", "CandidateKey", "CreatedDateKey", "VersionNumber"]]

df_test.drop_duplicates(inplace=True)
df_test.reset_index(inplace=True, drop=True)
df_test["TestKey"] = df_test.index + max_key + 1

df_test.head()

C:\Users\monad\AppData\Local\Temp\ipykernel_26600\1446861826.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.drop_duplicates(inplace=True)
C:\Users\monad\AppData\Local\Temp\ipykernel_26600\1446861826.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["TestKey"] = df_test.index + max_key + 1


,TestID,Test,CandidateKey,CreatedDateKey,VersionNumber,TestKey
0,1,PAQ,38114181.0,20190123,v1.0,6921
1,2,PAQ,38046331.0,20190110,v1.0,6922
2,3,PAQ,38114191.0,20190128,v1.0,6923
3,7,PAQ,37975171.0,20181219,v1.0,6924
4,8,PAQ,38046301.0,20190110,v1.0,6925


## FactQuestionSJT

In [21]:
df_question = df_paq.drop(columns=["CandidateKey", "CreatedDateKey", "ModifiedDateKey", "VersionNumber"])
df_question["QuestionKey"] = df_question.index + 1
df_question.info()

<class 'pandas.core.frame.DataFrame'>
Index: 451080 entries, 0 to 955124
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   InstanceID      451080 non-null  int64 
 1   LeftStatement   451080 non-null  object
 2   RightStatement  451080 non-null  object
 3   AnswerVal       451080 non-null  int32 
 4   Test            451080 non-null  object
 5   TestID          451080 non-null  int64 
 6   QuestionKey     451080 non-null  int64 
dtypes: int32(1), int64(3), object(3)
memory usage: 25.8+ MB


In [22]:
df_question = pd.merge(df_question, df_test[["TestKey", "TestID"]], on="TestID", how="left")
df_question.set_index('QuestionKey', inplace=True)
df_question.head()

,InstanceID,LeftStatement,RightStatement,AnswerVal,Test,TestID,TestKey
QuestionKey,,,,,,,
1,1,BEW_1,BEH_1,1,PAQ,1,6921
2,1,REA_1,OPT_1,5,PAQ,1,6921
3,1,REG_1,AFW_1,5,PAQ,1,6921
4,1,RUS_1,ENE_1,5,PAQ,1,6921
5,1,OVE_1,GED_1,5,PAQ,1,6921


In [23]:
df_test.drop(columns=["TestID"], inplace=True)
df_question.drop(columns=["TestID"], inplace=True)

C:\Users\monad\AppData\Local\Temp\ipykernel_26600\525301613.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.drop(columns=["TestID"], inplace=True)


## To csv 

In [24]:
df_test.to_csv('../decoded_data/PAQ/FactTest.csv', index=False)
df_question.to_csv('../decoded_data/PAQ/FactQuestionPAQ.csv')